使用facebook-opt-1.3b 的模型作为基础模型，插入lora进行微调，使得模型能够输出我们想要的答案

In [1]:
import torch
import random 

from util import TokenizerUtil

tokenizer = TokenizerUtil()

input_ids, attention_mask = tokenizer.encode('how are you', max_length=10)

input_ids, attention_mask, tokenizer.decode(input_ids)


libgomp: Invalid value for environment variable OMP_NUM_THREADS


(tensor([   0, 9178,   32,   47,    2,    1,    1,    1,    1,    1]),
 tensor([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]),
 '<s>how are you</s>')

加载数据集，打印下会发现又prompt，chosen，rejected，respose标准答案，四个部分组成

In [2]:
from datasets import load_dataset
from transformers import default_data_collator

dataset = load_dataset('json', data_files='dataset/train.json', split='train')
#2,4,4切分,取第0部分
dataset = dataset.select(range(15000))

dataset[0]

{'prompt': 'Human: context= CREATE TABLE table_name_93 (colours VARCHAR, house_name VARCHAR) question= What colours have a House Name of ogun? Assistant:',
 'chosen': 'SELECT colours FROM table_name_93 WHERE house_name = "ogun"',
 'rejected': '',
 'response': 'SELECT colours FROM table_name_93 WHERE house_name = "ogun"'}

In [3]:
def f(data):
    #随机生成两种回答
    if random.random() > 0.5:
        data['chosen'] = data['chosen'].swapcase() # 将字符串大小写翻转
    data = data['prompt'] + data['chosen']

    input_ids, attention_mask = tokenizer.encode(data) # 转化成编码 和 mask序列

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': input_ids.clone()
    }
    


In [4]:
dataset.column_names

['prompt', 'chosen', 'rejected', 'response']

这里remove_columns=dataset.column_names 表示去掉所有行列， 然后应用map函数对dataset中所有数据使用f函数， 即最后每个数据都会转化成这三个字段

In [5]:
dataset = dataset.map(f, remove_columns=dataset.column_names)

loader = torch.utils.data.DataLoader(dataset,
                                     collate_fn=default_data_collator,
                                     batch_size=2,
                                     shuffle=True,
                                     drop_last=True)

In [6]:
len(loader), next(iter(loader))

(7500,
 {'input_ids': tensor([[    0, 33837,    35,  ...,     1,     1,     1],
          [    0, 33837,    35,  ...,     1,     1,     1]]),
  'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0]]),
  'labels': tensor([[    0, 33837,    35,  ...,     1,     1,     1],
          [    0, 33837,    35,  ...,     1,     1,     1]])})

这里在model_actor 里面插入lora层

In [7]:
# 减少参数的量， 因此插入lora
from transformers import AutoModelForCausalLM
import lora
model_actor = AutoModelForCausalLM.from_pretrained(
    '/root/autodl-tmp/StudyLLM/02. Deepseed/opt-1.3b',
    torch_dtype=torch.bfloat16
)

lora.insert(model_actor)
lora.count_params(model_actor)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

{'count_require': 2.21044736, 'count_all': 14.29004288, 'ratio': 0.15468444556549854}


In [8]:
from transformers import get_scheduler
from accelerate import Accelerator

# 该函数就是将参数进行分组， lora的参数用5e-4学习率，actor其他参数使用默认学习率
def f():
    params = []
    params_lora = []
    for name, param in model_actor.named_parameters():
        if not param.requires_grad:
            continue

        if 'lora_A' in name or 'lora_B' in name:
            params_lora.append(param)
            continue

        params.append(param)

    return [{
        'params': params,
        'weight_decay': 0.0,
    }, {
        'params': params_lora,
        'weight_decay': 0.0,
        'lr': 5e-4
    }]


optimizer = torch.optim.Adam(f(), lr=1e-3, betas=(0.9, 0.95))

# 学习率调度器， 100轮内将学习率调整
scheduler = get_scheduler(name='cosine',
                          optimizer=optimizer,
                          num_warmup_steps=0,
                          num_training_steps=100)

# 加速组件， 其中梯度累积，64次梯度累计才更新一次梯度
accelerator = Accelerator(gradient_accumulation_steps=64,
                          mixed_precision='bf16') ## 半精度：把数据从32位减到16位，速度翻倍，显存减半

# 将模型，数据集，优化器，学习率调度器，放到加速组件变成加速组件版本的
model_actor, loader, optimizer, scheduler = accelerator.prepare(
    model_actor, loader, optimizer, scheduler)



In [9]:
model_actor.train()

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 2048, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 2048)
      (final_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Lora(
              (linear): Linear(in_features=2048, out_features=2048, bias=True)
            )
            (v_proj): Lora(
              (linear): Linear(in_features=2048, out_features=2048, bias=True)
            )
            (q_proj): Lora(
              (linear): Linear(in_features=2048, out_features=2048, bias=True)
            )
            (out_proj): Lora(
              (linear): Linear(in_features=2048, out_features=2048, bias=True)
            )
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
    

In [10]:
for i, data in enumerate(loader):
    # 加速器积累的方式进行梯度更新， 积累64次进行才进行梯度更新，这里loader里面的batch——size = 2， 所以一次放2条，然后32个batch后才更新梯度
    with accelerator.accumulate(model_actor):

        # out里面包装了很多内容：loss，logitis
        out = model_actor(**data)

        # 这一步是梯度计算 + 积累的关键
        accelerator.backward(out.loss)

        # 只有要进行梯度更新，即第32轮的时候sync_gradients 才是1进行剪切
        if accelerator.sync_gradients:
            accelerator.clip_grad_norm_(
                [i for i in model_actor.parameters() if i.requires_grad], 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    if (i + 1) % 100 == 0:
        # param_groups 是不同组的学习率
        lr = optimizer.param_groups[0]['lr']
        print(i, len(loader), out.loss.item(), lr)

        # out.logits.shape = [batch_size, lenth_sequence, vobalsize]
        logits = out.logits[0].argmax(1)
        print(tokenizer.decode(logits))

    if i == 2000:
        break

lora.merge(model_actor)
model_actor.save_pretrained('model/actor')

99 7500 12.335415840148926 0.0009997532801828658
-oid
.�ATE TABLE " (name (name (
ributesance_ARCHAR( date_ARCHAR, WHERE_ ' many people attended the event game? the Cardinals City Chiefs?
: CRE * FROM table_name_86 WHERE opponent = 'Kansasc_"" ANDI the the the the the the a the  the</s>
199 7500 3.9026005268096924 0.00099778098230154
i_
:�ATE_ (_name (name (
_name_ARCHAR ( home VARCHAR, WHERE_ CRE is is the venueue? the Kentucky? CRE: CRE venue_TEAM_ table_NAME_52 where HOMEARCHUE_ VWesternESTTERN OVAL" andIis__ist_:_otum:ireum_:ant_=umorig:ance=<pad>osumistanceant::::is__:otist:ut_ist:<pad>:istist:ol_<pad>al:umortum::ail::antantid_:ala:orum::id:otans_anceoralaosalisantotor::icorit_orol<pad>is姫alisist_ol_:oral::olantisantidic<pad>::<pad>_<pad>umant<pad>:um:alant:antalidumolum<pad>isIC_otumal<pad>um::u:_:<pad>ic:um:<pad>_umay_os<pad>al:ant:::um<pad>il:_idolistueant_<pad>:<pad><pad>or:or__otineol:ort:isonalor:isot::is::it_<pad>:a:aauma:um__um_on_<pad>ol<pad>sidor:::<pad>aala_::::ut<pad>:

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]